In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
from lightgbm import LGBMRegressor, LGBMClassifier
from xgboost import XGBClassifier
# 1. ĐỌC DỮ LIỆU
data = pd.read_csv(r'.\TrainingWiDS2021.csv')
target_col = 'diabetes_mellitus'

X = data.drop(columns=[target_col])
y = data[target_col]

# Chia tập Train/Test có phân tầng (stratify)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. TỰ ĐỘNG PHÂN LOẠI CÁC FEATURES
binary_cols = [col for col in X_train.columns if X_train[col].nunique() == 2]
categorical_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns 
                    if col not in binary_cols]
numerical_cols = [col for col in X_train.select_dtypes(include=['int64', 'float64']).columns 
                  if col not in binary_cols]

# 3. XÂY DỰNG PIPELINE TIỀN XỬ LÝ (KHÔNG IMPUTE, KHÔNG SCALE)
# - Binary: Map về số, NaNs tự động thành -1
binary_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1)

# - Categorical: One-Hot, bỏ qua giá trị lạ và NaN
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# - Numerical: Không biến đổi, để mô hình tự xử lý NaN
numerical_transformer = 'passthrough'

preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_transformer, binary_cols),
    ('cat', categorical_transformer, categorical_cols),
    ('num', numerical_transformer, numerical_cols)
])

# 4. CHỌN MÔ HÌNH VÀ HUẤN LUYỆN
# HistGradientBoosting tự động xử lý missing values, hỗ trợ class_weight
model = LGBMClassifier(n_estimators=30, random_state=42, device_type='gpu', class_weight='balanced',verbose=-1 )

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

pipeline.fit(X_train, y_train)

# 5. DỰ ĐOÁN VÀ ĐÁNH GIÁ
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print("--- ĐÁNH GIÁ MÔ HÌNH HISTGRADIENTBOOSTING ---")
print("\n1. Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n2. Classification Report:\n", classification_report(y_test, y_pred))
print(f"3. ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"4. PR-AUC (Average Precision): {average_precision_score(y_test, y_pred_proba):.4f}")

[LightGBM] [Info] Number of positive: 22521, number of negative: 81604
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 26484
[LightGBM] [Info] Number of data points in the train set: 104125, number of used features: 212
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


Exception ignored on calling ctypes callback function: <function _log_callback at 0x000002303B9C76D0>
Traceback (most recent call last):
  File "c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\lightgbm\basic.py", line 289, in _log_callback
    _log_native(str(msg.decode("utf-8")))
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb0 in position 163: invalid start byte
Exception ignored on calling ctypes callback function: <function _log_callback at 0x000002303B9C76D0>
Traceback (most recent call last):
  File "c:\Users\hi\miniconda3\envs\phan_tich_thuc_nghiem\lib\site-packages\lightgbm\basic.py", line 289, in _log_callback
    _log_native(str(msg.decode("utf-8")))
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x9a in position 170: invalid start byte


: 

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# Import LightGBM thay cho HistGradientBoostingClassifier để dùng GPU
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from lightgbm import LGBMRegressor, LGBMClassifier

# ---------------------------------------------------------
# HÀM BỔ TRỢ XỬ LÝ ĐẶC TRƯNG (FEATURE ENGINEERING)
# ---------------------------------------------------------
def extract_measurement_counts(df):
    df_out = df.copy()
    base_feats = [col.replace('_max', '') for col in df.columns if col.endswith('_max')]
    
    for base in base_feats:
        apache_name = base.replace('d1_', '').replace('h1_', '') + '_apache'
        cols_to_check = [f"{base}_max", f"{base}_min", apache_name]
        existing_cols = [c for c in cols_to_check if c in df.columns]
        
        if len(existing_cols) > 1:
            count_col = f"{base}_measure_count"
            df_out[count_col] = df_out[existing_cols].nunique(axis=1, dropna=True).astype(str)
            
            mean_by_count = df_out.groupby(count_col)[existing_cols[0]].transform('mean')
            df_out[f"{base}_shifted"] = df_out[existing_cols[0]] - mean_by_count
            
    return df_out

def extract_binned_stats(df, num_cols, n_bins=5):
    df_out = df.copy()
    for col in num_cols:
        bin_col = f"{col}_bin"
        df_out[bin_col] = pd.qcut(df_out[col], q=n_bins, duplicates='drop').astype(str)
        
        group_mean = df_out.groupby(bin_col)[col].transform('mean')
        group_std = df_out.groupby(bin_col)[col].transform('std')
        df_out[f"{col}_dist_std_from_mean"] = (df_out[col] - group_mean) / (group_std + 1e-8)
        
    return df_out

def drop_high_correlation(X_train, X_test, threshold=0.95):
    """ĐÃ FIX LỖI: Chỉ tính tương quan trên các cột dạng số (numeric)"""
    num_X_train = X_train.select_dtypes(include=[np.number])
    corr_matrix = num_X_train.corr(method='spearman').abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    
    return X_train.drop(columns=to_drop), X_test.drop(columns=to_drop)

# ---------------------------------------------------------
# 1. ĐỌC VÀ CHUẨN BỊ DỮ LIỆU
# ---------------------------------------------------------
data = pd.read_csv(r'.\TrainingWiDS2021.csv')
target_col = 'diabetes_mellitus'

X = data.drop(columns=[target_col, 'encounter_id', 'hospital_id', 'icu_id'], errors='ignore')
y = data[target_col]

key_num_cols = ['age', 'bmi', 'glucose_apache', 'd1_glucose_max']
key_num_cols = [c for c in key_num_cols if c in X.columns]

X = extract_measurement_counts(X)
X = extract_binned_stats(X, key_num_cols)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ---------------------------------------------------------
# 2. FEATURE ENGINEERING TRÊN TẬP TRAIN / TEST
# ---------------------------------------------------------
num_cols_all = X_train.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols_all:
    min_val, max_val = X_train[col].min(), X_train[col].max()
    X_train[col] = X_train[col].clip(lower=min_val, upper=max_val)
    X_test[col] = X_test[col].clip(lower=min_val, upper=max_val)

X_train, X_test = drop_high_correlation(X_train, X_test, threshold=0.98)

# ---------------------------------------------------------
# 3. TỰ ĐỘNG PHÂN LOẠI CÁC FEATURES
# ---------------------------------------------------------
binary_cols = [col for col in X_train.columns if X_train[col].nunique() == 2]
categorical_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns if col not in binary_cols]
numerical_cols = [col for col in X_train.select_dtypes(include=['int64', 'float64']).columns if col not in binary_cols]

# ---------------------------------------------------------
# 4. XÂY DỰNG PIPELINE TIỀN XỬ LÝ (GPU ACCELERATED IMPUTER)
# ---------------------------------------------------------
binary_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-1)
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Thêm tham số device_type='gpu'
numerical_transformer = Pipeline(steps=[
    ('imputer', IterativeImputer(
        estimator=LGBMRegressor(n_estimators=30, random_state=42, device_type='gpu',verbose=-1 ), 
        max_iter=5, 
        random_state=42))
])

preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_transformer, binary_cols),
    ('cat', categorical_transformer, categorical_cols),
    ('num', numerical_transformer, numerical_cols)
])

# ---------------------------------------------------------
# 5. CHỌN MÔ HÌNH VÀ HUẤN LUYỆN (GPU ACCELERATED CLASSIFIER)
# ---------------------------------------------------------
# Sử dụng LGBMClassifier thay cho HistGradientBoosting và bật GPU
model = LGBMClassifier(n_estimators=30, random_state=42, device_type='gpu', class_weight='balanced')

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

pipeline.fit(X_train, y_train)

# ---------------------------------------------------------
# 6. DỰ ĐOÁN VÀ ĐÁNH GIÁ
# ---------------------------------------------------------
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print("--- ĐÁNH GIÁ MÔ HÌNH LIGHTGBM (GPU) ---")
print("\n1. Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n2. Classification Report:\n", classification_report(y_test, y_pred))
print(f"3. ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"4. PR-AUC (Average Precision): {average_precision_score(y_test, y_pred_proba):.4f}")

D:\ml_cache\temp\ipykernel_1832\2740095983.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f"{base}_shifted"] = df_out[existing_cols[0]] - mean_by_count
D:\ml_cache\temp\ipykernel_1832\2740095983.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[count_col] = df_out[existing_cols].nunique(axis=1, dropna=True).astype(str)
D:\ml_cache\temp\ipykernel_1832\2740095983.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform

ValueError: could not convert string to float: 'Other/Unknown'